# Choropleth Map – Gold Trade per Country
Visualisiert den Goldhandel (Import/Export/Total) pro Land mit Plotly.

In [15]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../../data/processed/precious_metals_trade_1988_2024.csv")

# Nur Gold
gold = df[df["metal_group"] == "Gold"].copy()
print(gold.shape)
gold.head(3)

(76235, 7)


,country,year,metal_group,flow,trade_value_usd,weight_kg,Headline CPI
4143,Albania,2013,Gold,Import,36856,970.0,1.934749
4148,Albania,2011,Gold,Import,12654,0.0,3.414678
4153,Albania,2001,Gold,Import,1467,32.0,3.104656


In [16]:
# Ländernamen auf ISO-3 mappen (Plotly braucht ISO-Alpha-3)
# pycountry als Hilfsmittel
try:
    import pycountry
    def to_iso3(name):
        try:
            return pycountry.countries.lookup(name).alpha_3
        except LookupError:
            return None
except ImportError:
    # Fallback: manuelle Mapping-Tabelle für häufigste Länder
    manual_map = {
        "Hong Kong SAR, China": "HKG",
        "EU-28": None,  # kein ISO-3
        "India": "IND",
        "China": "CHN",
        "Germany": "DEU",
        "Canada": "CAN",
        "Italy": "ITA",
        "Australia": "AUS",
        "Japan": "JPN",
        "France": "FRA",
        "United States": "USA",
        "Switzerland": "CHE",
        "United Kingdom": "GBR",
        "South Africa": "ZAF",
        "Russia": "RUS",
        "Brazil": "BRA",
        "Mexico": "MEX",
        "Turkey": "TUR",
        "UAE": "ARE",
        "Saudi Arabia": "SAU",
        "Korea, Rep.": "KOR",
        "Singapore": "SGP",
        "Thailand": "THA",
        "Indonesia": "IDN",
        "Malaysia": "MYS",
        "Peru": "PER",
        "Ghana": "GHA",
        "Tanzania": "TZA",
        "Mali": "MLI",
        "Burkina Faso": "BFA",
        "Argentina": "ARG",
        "Chile": "CHL",
        "Colombia": "COL",
        "Venezuela": "VEN",
        "Kazakhstan": "KAZ",
        "Uzbekistan": "UZB",
        "Mongolia": "MNG",
        "Papua New Guinea": "PNG",
        "Philippines": "PHL",
        "Myanmar": "MMR",
        "Vietnam": "VNM",
        "Pakistan": "PAK",
        "Bangladesh": "BGD",
        "Sri Lanka": "LKA",
        "Egypt": "EGY",
        "Morocco": "MAR",
        "Algeria": "DZA",
        "Nigeria": "NGA",
        "Ethiopia": "ETH",
        "Kenya": "KEN",
        "Zimbabwe": "ZWE",
        "Zambia": "ZMB",
        "Congo, Dem. Rep.": "COD",
        "Spain": "ESP",
        "Netherlands": "NLD",
        "Belgium": "BEL",
        "Austria": "AUT",
        "Sweden": "SWE",
        "Poland": "POL",
        "Czech Republic": "CZE",
        "Portugal": "PRT",
        "Greece": "GRC",
        "Romania": "ROU",
        "Hungary": "HUN",
        "Ukraine": "UKR",
        "United Arab Emirates": "ARE",
        "Iran": "IRN",
        "Iraq": "IRQ",
        "Israel": "ISR",
        "New Zealand": "NZL",
        "Finland": "FIN",
        "Denmark": "DNK",
        "Norway": "NOR",
        "Luxembourg": "LUX",
        "Cyprus": "CYP",
        "Malta": "MLT",
        "Albania": "ALB",
        "Ecuador": "ECU",
    }
    def to_iso3(name):
        return manual_map.get(name, None)

print("ISO-3 Mapping bereit")

ISO-3 Mapping bereit


In [17]:
# --- Daten aggregieren ---

# Gesamthandelsvolumen pro Land (Import + Export summiert)
gold_total = (
    gold.groupby("country")["trade_value_usd"]
    .sum()
    .reset_index()
    .rename(columns={"trade_value_usd": "total_trade_usd"})
)

# ISO-3 Codes hinzufügen
gold_total["iso3"] = gold_total["country"].apply(to_iso3)

# Länder ohne ISO-3 rausfiltern (z.B. EU-28)
gold_map = gold_total.dropna(subset=["iso3"]).copy()

# In Milliarden USD umrechnen für bessere Lesbarkeit
gold_map["trade_bn_usd"] = gold_map["total_trade_usd"] / 1e9

print(f"Länder mit ISO-3: {len(gold_map)} von {len(gold_total)}")
gold_map.sort_values("trade_bn_usd", ascending=False).head(10)

Länder mit ISO-3: 68 von 245


,country,total_trade_usd,iso3,trade_bn_usd
108,"Hong Kong SAR, China",976790531209,HKG,976.790531
111,India,763775703012,IND,763.775703
216,Switzerland,739522169254,CHE,739.522169
45,China,594887942073,CHN,594.887942
234,United Kingdom,500618068495,GBR,500.618068
39,Canada,345231780676,CAN,345.231781
233,United Arab Emirates,328271996784,ARE,328.271997
11,Australia,327528929397,AUS,327.528929
94,Germany,242292259707,DEU,242.292260
117,Italy,223818325136,ITA,223.818325


In [18]:
fig = px.choropleth(
    gold_map,
    locations="iso3",
    color="trade_bn_usd",
    hover_name="country",
    hover_data={"trade_bn_usd": ":.2f", "iso3": False},
    color_continuous_scale="YlOrRd",
    range_color=(0, gold_map["trade_bn_usd"].quantile(0.95)),
    labels={"trade_bn_usd": "Goldhandel (Mrd. USD)"},
    title="Goldhandel pro Land (1988–2024, kumulativ)",
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type="natural earth",
    ),
    coloraxis_colorbar=dict(
        title="Mrd. USD",
        tickformat=".0f",
    ),
    margin=dict(l=0, r=0, t=50, b=0),
)

fig.show()

In [19]:
gold_flow = (
    gold[gold["flow"].isin(["Import", "Export"])]
    .groupby(["country", "flow"])["trade_value_usd"]
    .sum()
    .reset_index()
)
gold_flow["iso3"] = gold_flow["country"].apply(to_iso3)
gold_flow = gold_flow.dropna(subset=["iso3"])
gold_flow["trade_bn_usd"] = gold_flow["trade_value_usd"] / 1e9

fig2 = px.choropleth(
    gold_flow,
    locations="iso3",
    color="trade_bn_usd",
    hover_name="country",
    facet_col="flow",
    color_continuous_scale="YlOrRd",
    range_color=(0, gold_flow["trade_bn_usd"].quantile(0.95)),
    labels={"trade_bn_usd": "Goldhandel (Mrd. USD)"},
    title="Gold Import vs. Export pro Land (1988–2024)",
)

fig2.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type="natural earth"),
    margin=dict(l=0, r=0, t=60, b=0),
)

fig2.show()

In [20]:
# --- Choropleth Map: Animated per Year ---

gold_year = (
    gold[gold["flow"].isin(["Import", "Export"])]
    .groupby(["country", "year"])["trade_value_usd"]
    .sum()
    .reset_index()
)
gold_year["iso3"] = gold_year["country"].apply(to_iso3)
gold_year = gold_year.dropna(subset=["iso3"])
gold_year["trade_bn_usd"] = gold_year["trade_value_usd"] / 1e9

gold_year = gold_year.sort_values("year")

fig3 = px.choropleth(
    gold_year,
    locations="iso3",
    color="trade_bn_usd",
    hover_name="country",
    animation_frame="year",
    color_continuous_scale="YlOrRd",
    range_color=(0, gold_year["trade_bn_usd"].quantile(0.98)),
    labels={"trade_bn_usd": "Goldhandel (Mrd. USD)"},
    title="Goldhandel pro Land und Jahr (animiert)",
)

fig3.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type="natural earth"),
    margin=dict(l=0, r=0, t=60, b=0),
)

fig3.show()